## Particionamento

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types as t
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder\
        .config("spark.master", "local[3]")\
        .config("spark.app.name", "AprendendoDF")\
        .config("spark.executor.memory", "450m")\
        .config("spark.cores.max","3")\
        .config("spark.sql.shuffle.partitions", "20")\
        .getOrCreate()
spark

In [3]:
schema = t.StructType([
    t.StructField("longitude", t.DoubleType(), nullable=True),
    t.StructField("latitude", t.DoubleType(), nullable=True),
    t.StructField("housing_median_age", t.DoubleType(), nullable=True),
    t.StructField("total_rooms", t.DoubleType(), nullable=True),
    t.StructField("total_bedrooms", t.DoubleType(), nullable=True),
    t.StructField("population", t.DoubleType(), nullable=True),
    t.StructField("households", t.DoubleType(), nullable=True),
    t.StructField("median_income", t.DoubleType(), nullable=True),
    t.StructField("median_house_value", t.DoubleType(), nullable=True),
    t.StructField("ocean_proximity", t.StringType(), nullable=True),
])
schema

StructType([StructField('longitude', DoubleType(), True), StructField('latitude', DoubleType(), True), StructField('housing_median_age', DoubleType(), True), StructField('total_rooms', DoubleType(), True), StructField('total_bedrooms', DoubleType(), True), StructField('population', DoubleType(), True), StructField('households', DoubleType(), True), StructField('median_income', DoubleType(), True), StructField('median_house_value', DoubleType(), True), StructField('ocean_proximity', StringType(), True)])

In [4]:
data = spark.read\
    .format("csv")\
    .csv(
        path="C:\\Users\\mateu\\Documents\\Norton\\Projetos GIT\\learn-databricks\\dataset\\housing.csv",
        schema=schema,
        sep=',',
        header=True
    )

In [5]:
data.rdd.getNumPartitions()

1

In [6]:
data = data.repartition(4,"ocean_proximity")

In [7]:
data.rdd.getNumPartitions()

4

In [8]:
data.groupBy("ocean_proximity").count().show()

+---------------+-----+
|ocean_proximity|count|
+---------------+-----+
|     NEAR OCEAN| 2658|
|      <1H OCEAN| 9136|
|         INLAND| 6551|
|         ISLAND|    5|
|       NEAR BAY| 2290|
+---------------+-----+



In [9]:
col = data.columns
col.remove('ocean_proximity')
col

['longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income',
 'median_house_value']

In [10]:
data = data.select(
    *col, 
    F.lower(
        F.regexp_replace(
            F.regexp_replace(F.col("ocean_proximity"), "<","More"), " ", "_"
        )
    ).alias("ocean_proximity")   
)
data.show(2)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -124.17|    41.8|              16.0|     2739.0|         480.0|    1259.0|     436.0|       3.7557|          109400.0|     near_ocean|
|   -124.3|    41.8|              19.0|     2672.0|         552.0|    1298.0|     478.0|       1.9797|           85800.0|     near_ocean|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
only showing top 2 rows


No PySpark, **particionamento** é um dos conceitos mais cruciais para performance. Como o Spark é um framework de computação distribuída, ele divide os dados em pedaços menores (partições) para que diferentes nós do cluster possam processá-los em paralelo.

Se você tem partições de menos, seu cluster fica ocioso. Se tem partições demais, o Spark gasta mais tempo gerenciando tarefas do que processando dados.

Aqui está um guia prático de como funcionam e como gerenciar partições no PySpark:

---

## 1. Como ver as partições atuais

Para saber em quantas partes seu DataFrame está dividido, use o método `.rdd.getNumPartitions()`:

```python
# Ver o número de partições
print(df.rdd.getNumPartitions())

```

---

## 2. Re-particionamento: `repartition()` vs `coalesce()`

Quando você precisa alterar o número de partições de um DataFrame, o PySpark oferece duas funções principais. Escolher a errada pode travar seu pipeline.

### `repartition(n)`

* **Como funciona:** Faz um *Full Shuffle* (redistribui todos os dados pela rede entre os nós do cluster).
* **Quando usar:** Quando você quer **aumentar** o número de partições ou quando quer garantir que os dados fiquem distribuídos uniformemente (evitando *data skew*).
* **Custo:** Alto (envolve muita troca de dados na rede).

```python
# Aumentando para 10 partições
df_repartitioned = df.repartition(10)

```

### `coalesce(n)`

* **Como funciona:** Reduz o número de partições sem fazer um shuffle completo. Ele apenas junta partições existentes no mesmo nó.
* **Quando usar:** Quando você quer **diminuir** o número de partições (ex: antes de salvar um arquivo para não gerar milhares de micro-arquivos).
* **Custo:** Baixo (muito mais rápido que o `repartition`).

```python
# Diminuindo para 2 partições de forma eficiente
df_coalesced = df.coalesce(2)

```

---

## 3. Particionamento por Coluna (Criando Grupos)

Você também pode passar o nome de uma coluna para o `repartition()`. O Spark vai usar o hash dessa coluna para mandar todas as linhas com o mesmo valor para a mesma partição.

```python
# Garante que todos os dados do mesmo país fiquem na mesma partição
df_por_pais = df.repartition("pais")

```

> 💡 **Utilidade:** Isso é excelente para otimizar operações subsequentes de `groupBy` ou `join`, pois o Spark não precisará mover os dados pela rede novamente.

---

## 4. Gravando Dados Particionados (No Disco)

Ao salvar os dados em formatos como Parquet ou CSV, você pode particionar os dados em pastas físicas baseadas em uma coluna (geralmente datas, regiões, etc.).

```python
# Salva os dados criando uma estrutura de pastas: /caminho/ano=2026/mes=06/
df.write \
  .partitionBy("ano", "mes") \
  .mode("overwrite") \
  .parquet("caminho/do/output")

```

---

## Summary: Qual escolher?

| Operação | `coalesce` | `repartition` | `partitionBy` (ao salvar) |
| --- | --- | --- | --- |
| **Objetivo** | Diminuir partições | Aumentar/Mudar partições | Organizar arquivos no disco |
| **Faz Shuffle?** | Não (geralmente) | Sim (sempre) | N/A (operação de I/O) |
| **Impacto de Performance** | Leve / Rápido | Pesado / Custo de rede | Alto ganho para leituras futuras |

Qual é o cenário ou problema específico de performance que você está tentando resolver com partições hoje?